In [1]:
import sys
import subprocess

# Install
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'shapely', 'pandas', 'numpy'])

# Add user site-packages to path for this session
import site
sys.path.insert(0, site.getusersitepackages())

import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon

In [2]:
cellBoundaries = pd.read_csv("/scratch/pl2820/competition/train/ground_truth/cell_boundaries_train.csv")
spotsTrain = pd.read_csv("/scratch/pl2820/competition/train/ground_truth/spots_train.csv")

In [ ]:
def parse_polygon(row, z):
    xs = row[f'boundaryX_z{z}']
    ys = row[f'boundaryY_z{z}']
    if isinstance(xs, str):
        xs = list(map(float, xs.split(',')))
        ys = list(map(float, ys.split(',')))
    elif isinstance(xs, (int, float)):
        return None  # Not a polygon, skip
    
    coords = list(zip(xs, ys))
    if len(coords) >= 3:
        return Polygon(coords)
    return None

z_planes = [0, 1, 2, 3, 4]
polygons_by_z = {}
for z in z_planes:
    polys = []
    for idx, row in cellBoundaries.iterrows():
        poly = parse_polygon(row, z)
        if poly is not None and poly.is_valid:
            polys.append((idx, poly))
    polygons_by_z[z] = polys
    print(f"z{z}: {len(polys)} valid polygons")

def find_containing_cell(spot_x, spot_y, spot_z):
    z = int(round(spot_z))
    if z not in polygons_by_z:
        return -1
    pt = Point(spot_x, spot_y)
    for cell_idx, poly in polygons_by_z[z]:
        if poly.contains(pt):
            return cell_idx
    return -1

spotsTrain['cell_boundary_index'] = spotsTrain.apply(
    lambda row: find_containing_cell(row['x'], row['y'], row['global_z']),
    axis=1
)

output = spotsTrain[['barcode_id', 'fov', 'x', 'y', 'global_z', 'target_gene', 'cell_boundary_index']]
output.to_csv('/scratch/ccn280/spots_with_cell_assignment.csv', index=False)
print(f"Done. {(output['cell_boundary_index'] != -1).sum()} / {len(output)} spots assigned to a cell.")

z0: 3080 valid polygons
z1: 3610 valid polygons
z2: 3774 valid polygons
z3: 3537 valid polygons
z4: 3069 valid polygons
